# MNIST Experiment: Wide Feature CNN (Denoising Convolutional Autoencoder & Classification Evaluation)

Trains a **Denoising Convolutional Autoencoder** with a joint classification head on MNIST subsets (2500 and 5000 images) for 50 and 100 epochs.

### Architecture Highlights:
- **Encoder**: Multi-stage convolutions (1 $\rightarrow$ 32 $\rightarrow$ 32 $\rightarrow$ MaxPool $\rightarrow$ 64 $\rightarrow$ 64 $\rightarrow$ MaxPool $\rightarrow$ 128 $\rightarrow$ 128) with BatchNorm & ReLU.
- **Decoder**: Progressive transposed convolutions and convolutions restoring image back to exactly `[1, 28, 28]` with `Sigmoid`.
- **Classification Head**: Latent representation classifier predicting digit classes 0–9.
- **Dual Evaluation**: Evaluated before noise (**Clean Test Set**) and after noise (**15% Gaussian Noisy Test Set**).
- **8 Confusion Matrices**: Complete side-by-side comparison across all 4 experiments (4 runs $\times$ 2 evaluations = 8 matrices).
- **Complete Epoch CSV Logs**: All per-epoch metrics logged for epochs 1–50 and 1–100 (`results/epoch_logs.csv` and per-run CSVs).


In [21]:
%pip install -q torch torchvision scikit-learn pandas matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuration & Strict Assignment Rules

Strict assignment parameters:
- Dataset sizes: 2,500 and 5,000
- Splits: 70% Train, 10% Validation, 20% Test
- Seed: 42 across all experiments
- Test Noise: 15% Gaussian noise applied strictly to the test set
- Learning rate: 0.01 (SGD), Batch size: 64
- Epochs: 50 and 100


In [22]:
from pathlib import Path
import random
import copy
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Strict assignment settings
DATASET_SIZES = [2500, 5000]      # Subset sizes: 2500, 5000
EPOCHS_LIST = [50, 100]           # Epochs: 50, 100
SEED = 42                         # Random seed for reproducibility
TRAIN_RATIO = 0.70                # 70% Train
TEST_RATIO = 0.20                 # 20% Test
VAL_RATIO = 0.10                  # 10% Validation
NOISE_LEVEL = 0.15                # 15% input noise ONLY on test set
LEARNING_RATE = 0.01              # Fixed SGD learning rate
BATCH_SIZE = 64                   # DataLoader batch size
MODEL_NAME = "Denoising Convolutional Autoencoder"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    torch.set_num_threads(8)
print("Device:", DEVICE)

def safe_to_csv(df, path):
    try:
        df.to_csv(path, index=False)
        print(f"  Saved CSV to: {path}")
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_latest{path.suffix}")
        df.to_csv(fallback, index=False)
        print(f"  [Notice] '{path.name}' is currently open in Excel or another program. Saved to '{fallback.name}' instead.")


Device: cpu


## 2. Reproducibility & Seeding

In [23]:
def set_seed(seed=SEED):
    """Set random seeds for Python, NumPy, and PyTorch for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


## 3. Dataset Loading, Stratified Splitting & Reference Digits

In [24]:
# Load complete MNIST (60,000 train + 10,000 test = 70,000 total)
mnist_train = datasets.MNIST(DATA_ROOT, train=True, download=True)
mnist_test = datasets.MNIST(DATA_ROOT, train=False, download=True)

full_images = torch.cat([mnist_train.data, mnist_test.data], dim=0).float().div(255.0).unsqueeze(1)
full_labels = torch.cat([mnist_train.targets, mnist_test.targets], dim=0)
print(f"Total MNIST images loaded: {len(full_labels)}")

def get_reference_digit_images(images=full_images, labels=full_labels):
    reference_images = {}
    for img, lbl in zip(images, labels):
        digit = int(lbl)
        if digit not in reference_images:
            reference_images[digit] = img.squeeze().numpy()
        if len(reference_images) == 10:
            break
    return reference_images

reference_images = get_reference_digit_images()

def get_data(n, seed=SEED):
    X, _, y, _ = train_test_split(
        full_images, full_labels,
        train_size=n,
        stratify=full_labels,
        random_state=seed
    )
    Xtr, Xt, ytr, yt = train_test_split(
        X, y,
        train_size=TRAIN_RATIO,
        stratify=y,
        random_state=seed
    )
    val_rel = VAL_RATIO / (TEST_RATIO + VAL_RATIO)
    Xva, Xte, yva, yte = train_test_split(
        Xt, yt,
        train_size=val_rel,
        stratify=yt,
        random_state=seed
    )
    # 15% Gaussian noise applied ONLY to the test set
    noise_gen = torch.Generator().manual_seed(seed)
    noise = torch.randn(Xte.shape, generator=noise_gen) * NOISE_LEVEL
    Xte_noisy = (Xte + noise).clamp(0.0, 1.0)
    return (Xtr, ytr), (Xva, yva), (Xte, yte), Xte_noisy


Total MNIST images loaded: 70000


## 4. Aligned Training Data Augmentation

Applies identical geometric affine transformations to both the input and target before adding noise to the input. This ensures the target remains geometrically aligned with the reconstructed image.

In [25]:
class TrainAugmentation:
    def __init__(self, noise_std=0.15):
        self.noise_std = noise_std
        self.affine = transforms.RandomAffine(degrees=7, translate=(0.06, 0.06))

    def __call__(self, x_clean):
        x_aug = self.affine(x_clean)
        noise = torch.randn_like(x_aug) * self.noise_std
        x_noisy = (x_aug + noise).clamp(0.0, 1.0)
        return x_noisy, x_aug


## 5. Denoising Convolutional Autoencoder Architecture

- **Encoder**: Multi-stage Conv2d with BatchNorm and ReLU downsampling to latent shape `[B, 128, 7, 7]`.
- **Decoder**: ConvTranspose2d and Conv2d layers reconstructing clean `[B, 1, 28, 28]` with Sigmoid.
- **Classification Head**: Dense classification head predicting digit classes 0–9 for generating the 8 confusion matrices.


In [26]:
class DualHeadAutoencoder(nn.Module):
    def __init__(self):
        super(DualHeadAutoencoder, self).__init__()

        # --- ENCODER ---
        self.enc_conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.enc_bn1 = nn.BatchNorm2d(32)
        self.enc_conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False)
        self.enc_bn2 = nn.BatchNorm2d(32)
        self.enc_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14

        self.enc_conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.enc_bn3 = nn.BatchNorm2d(64)
        self.enc_conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.enc_bn4 = nn.BatchNorm2d(64)
        self.enc_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7

        self.enc_conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.enc_bn5 = nn.BatchNorm2d(128)
        self.enc_conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False)
        self.enc_bn6 = nn.BatchNorm2d(128)

        # --- DECODER ---
        self.dec_up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2, bias=False)  # 7x7 -> 14x14
        self.dec_bn1 = nn.BatchNorm2d(64)
        self.dec_conv1 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.dec_bn2 = nn.BatchNorm2d(64)

        self.dec_up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2, bias=False)  # 14x14 -> 28x28
        self.dec_bn3 = nn.BatchNorm2d(32)
        self.dec_conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False)
        self.dec_bn4 = nn.BatchNorm2d(32)

        self.dec_out = nn.Conv2d(32, 1, kernel_size=3, padding=1)

        # --- CLASSIFIER HEAD ---
        self.cls_pool = nn.MaxPool2d(kernel_size=2, stride=2)  # 7x7 -> 3x3
        self.cls_fc1 = nn.Linear(128 * 3 * 3, 128, bias=False)
        self.cls_bn = nn.BatchNorm1d(128)
        self.cls_drop = nn.Dropout(0.40)
        self.cls_fc2 = nn.Linear(128, 10)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = F.relu(self.enc_bn1(self.enc_conv1(x)))
        x = F.relu(self.enc_bn2(self.enc_conv2(x)))
        x = self.enc_pool1(x)

        x = F.relu(self.enc_bn3(self.enc_conv3(x)))
        x = F.relu(self.enc_bn4(self.enc_conv4(x)))
        x = self.enc_pool2(x)

        x = F.relu(self.enc_bn5(self.enc_conv5(x)))
        h = F.relu(self.enc_bn6(self.enc_conv6(x)))

        d = F.relu(self.dec_bn1(self.dec_up1(h)))
        d = F.relu(self.dec_bn2(self.dec_conv1(d)))
        d = F.relu(self.dec_bn3(self.dec_up2(d)))
        d = F.relu(self.dec_bn4(self.dec_conv2(d)))
        recon = torch.sigmoid(self.dec_out(d))

        c = self.cls_pool(h)
        c = torch.flatten(c, 1)
        c = F.relu(self.cls_bn(self.cls_fc1(c)))
        c = self.cls_drop(c)
        logits = self.cls_fc2(c)
        return recon, logits

def get_model():
    return DualHeadAutoencoder().to(DEVICE)


## 6. Training & Evaluation Engine (Reconstruction & Classification)

In [27]:
def compute_psnr(mse):
    if mse <= 0.0:
        return 99.0
    return 10.0 * math.log10(1.0 / mse)

def compute_ssim_tensor(img1, img2):
    C1 = (0.01) ** 2
    C2 = (0.03) ** 2
    mu1 = F.avg_pool2d(img1, kernel_size=3, stride=1, padding=1)
    mu2 = F.avg_pool2d(img2, kernel_size=3, stride=1, padding=1)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    sigma1_sq = F.avg_pool2d(img1 ** 2, kernel_size=3, stride=1, padding=1) - mu1_sq
    sigma2_sq = F.avg_pool2d(img2 ** 2, kernel_size=3, stride=1, padding=1) - mu2_sq
    sigma12 = F.avg_pool2d(img1 * img2, kernel_size=3, stride=1, padding=1) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean().item()

def evaluate_model(model, X_input, X_target_clean, y_labels, batch_size=BATCH_SIZE):
    model.eval()
    dataset = TensorDataset(X_input, X_target_clean, y_labels)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    total_mse = 0.0
    total_cls_loss = 0.0
    correct = 0
    total = 0
    all_recons = []
    all_preds = []
    with torch.no_grad():
        for bx_in, bx_tgt, by in loader:
            bx_in = bx_in.to(DEVICE)
            bx_tgt = bx_tgt.to(DEVICE)
            by = by.to(DEVICE)
            recon, logits = model(bx_in)
            total_mse += F.mse_loss(recon, bx_tgt, reduction='sum').item()
            total_cls_loss += F.cross_entropy(logits, by, reduction='sum').item()
            preds = logits.argmax(dim=1)
            correct += (preds == by).sum().item()
            total += len(by)
            all_recons.append(recon.cpu())
            all_preds.extend(preds.cpu().numpy())
    all_recons = torch.cat(all_recons, dim=0)
    avg_mse = total_mse / (total * 28 * 28)
    avg_cls_loss = total_cls_loss / total
    accuracy = (correct / total) * 100.0
    psnr = compute_psnr(avg_mse)
    ssim = compute_ssim_tensor(all_recons, X_target_clean)
    return {
        "mse": avg_mse,
        "psnr": psnr,
        "ssim": ssim,
        "cls_loss": avg_cls_loss,
        "accuracy": accuracy,
        "predictions": np.array(all_preds),
        "reconstructions": all_recons
    }

def train_autoencoder(model, train_data, val_data, optimizer, epochs, batch_size=BATCH_SIZE):
    Xtr, ytr = train_data
    Xva, yva = val_data
    noise_va = torch.randn(Xva.shape) * NOISE_LEVEL
    Xva_noisy = (Xva + noise_va).clamp(0.0, 1.0)
    train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    augment = TrainAugmentation(noise_std=NOISE_LEVEL)
    history = {"epoch": [], "train_loss": [], "train_recon_mse": [], "train_accuracy": [], "val_loss": [], "val_recon_mse": [], "val_accuracy": [], "val_psnr": []}
    best_val_acc = -1.0
    best_weights = copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        model.train()
        running_total_loss, running_recon_mse, correct_tr, total_tr = 0.0, 0.0, 0, 0
        for bx_clean, by in train_loader:
            bx_noisy, bx_target = augment(bx_clean)
            bx_noisy, bx_target, by = bx_noisy.to(DEVICE), bx_target.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            recon, logits = model(bx_noisy)
            loss_recon = F.mse_loss(recon, bx_target)
            loss_cls = F.cross_entropy(logits, by)
            total_loss = 10.0 * loss_recon + loss_cls
            total_loss.backward()
            optimizer.step()
            bs = len(by)
            running_total_loss += total_loss.item() * bs
            running_recon_mse += loss_recon.item() * bs
            correct_tr += (logits.argmax(dim=1) == by).sum().item()
            total_tr += bs
        val_res = evaluate_model(model, Xva_noisy, Xva, yva, batch_size=batch_size)
        val_total_loss = 10.0 * val_res["mse"] + val_res["cls_loss"]
        history["epoch"].append(epoch)
        history["train_loss"].append(running_total_loss / total_tr)
        history["train_recon_mse"].append(running_recon_mse / total_tr)
        history["train_accuracy"].append((correct_tr / total_tr) * 100.0)
        history["val_loss"].append(val_total_loss)
        history["val_recon_mse"].append(val_res["mse"])
        history["val_accuracy"].append(val_res["accuracy"])
        history["val_psnr"].append(val_res["psnr"])
        if val_res["accuracy"] > best_val_acc:
            best_val_acc = val_res["accuracy"]
            best_weights = copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"  Epoch [{epoch:3d}/{epochs:3d}] | Train Loss: {history['train_loss'][-1]:.4f} (MSE: {history['train_recon_mse'][-1]:.4f}, Acc: {history['train_accuracy'][-1]:.2f}%) | Val Acc: {val_res['accuracy']:.2f}% | Val PSNR: {val_res['psnr']:.2f} dB (Best Val Acc: {best_val_acc:.2f}%)")
    model.load_state_dict(best_weights)
    return history, best_val_acc


## 7. Visualization Suite (8 Confusion Matrices, Learning Curves, 10 Digits & Comparison)

Defines all visualization functions conforming strictly to the assignment specifications:
1. `plot_all_four_charts`: 4-panel dashboard per experiment (Loss curve, Accuracy curve, 10 digits, Clean vs Noisy CMs).
2. `plot_ten_digits`: 10-digit reconstruction and predictions per experiment.
3. `plot_single_confusion_matrices`: Individual clean, noisy, and side-by-side confusion matrices per experiment.
4. `plot_all_eight_confusion_matrices`: Master 8-panel comparison figure across all 4 situations.
5. `plot_all_ten_digits`: Master 10-digit denoising comparison across all 4 situations.
6. `plot_learning_curves`: Training & validation loss, accuracy, MSE, and PSNR curves across all 4 situations.
7. `plot_final_evaluation`: Final evaluation accuracy comparison bar chart.

In [28]:
def plot_all_four_charts(artifact, output_dir=RESULTS_DIR):
    """Combined 4-panel dashboard saved as all_four_charts.png for each experiment."""
    size = artifact["dataset_size"]
    epochs = artifact["epochs"]
    hist = artifact["history"]
    Xte = artifact["Xte"]
    Xnoisy = artifact["Xte_noisy"]
    recons = artifact["reconstructions"]
    yte = artifact["yte"]
    preds = artifact["predictions"]
    cm_clean = artifact["cm_clean"]
    cm_noisy = artifact["cm_noisy"]

    run_folder = output_dir / f"mnist_{size}_{epochs}"
    run_folder.mkdir(parents=True, exist_ok=True)

    fig = plt.figure(figsize=(16, 20))
    grid = fig.add_gridspec(2, 2, height_ratios=[1, 2.2], hspace=0.20, wspace=0.18)

    # 1. Loss Curves
    ax_loss = fig.add_subplot(grid[0, 0])
    ep_range = hist["epoch"]
    ax_loss.plot(ep_range, hist["train_loss"], label="Training Loss", lw=1.8)
    ax_loss.plot(ep_range, hist["val_loss"], label="Validation Loss", lw=1.8)
    ax_loss.set_title("Training vs Validation Total Loss", fontweight="bold")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    # 2. Accuracy Curves
    ax_acc = fig.add_subplot(grid[0, 1])
    ax_acc.plot(ep_range, hist["train_accuracy"], label="Training Accuracy", lw=1.8)
    ax_acc.plot(ep_range, hist["val_accuracy"], label="Validation Accuracy", lw=1.8)
    ax_acc.set_title("Training vs Validation Accuracy", fontweight="bold")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy (%)")
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)

    # 3. 10 Digit Predictions & Reconstructions
    ax_pred_container = fig.add_subplot(grid[1, 0])
    ax_pred_container.set_title(
        "10 Digit Denoising (Original Clean | 15% Noisy Input | Reconstructed Clean Image)",
        fontweight="bold",
        fontsize=11,
        pad=8
    )
    ax_pred_container.axis("off")

    sample_idx = [(yte == d).nonzero()[0].item() for d in range(10)]
    subgrid = ax_pred_container.get_subplotspec().subgridspec(
        11, 3, height_ratios=[0.15] + [1] * 10, hspace=0.12, wspace=0.08
    )

    for col, header in enumerate(["Original Clean", "Test (15% Noise)", "Reconstructed Clean"]):
        h_ax = fig.add_subplot(subgrid[0, col])
        h_ax.text(0.5, 0.5, header, ha="center", va="center", fontsize=9, fontweight="bold")
        h_ax.axis("off")

    for i in range(10):
        idx_i = sample_idx[i]
        pred_digit = int(preds[idx_i])
        orig_img = Xte[idx_i, 0].numpy()
        noisy_img = Xnoisy[idx_i, 0].numpy()
        recon_img = recons[idx_i, 0].numpy()

        ax_orig = fig.add_subplot(subgrid[i + 1, 0])
        ax_noisy = fig.add_subplot(subgrid[i + 1, 1])
        ax_res = fig.add_subplot(subgrid[i + 1, 2])

        ax_orig.imshow(orig_img, cmap="gray", vmin=0, vmax=1)
        ax_noisy.imshow(noisy_img, cmap="gray", vmin=0, vmax=1)
        ax_res.imshow(recon_img, cmap="gray", vmin=0, vmax=1)

        ax_orig.set_xticks([])
        ax_orig.set_yticks([])
        ax_noisy.set_xticks([])
        ax_noisy.set_yticks([])
        ax_res.set_xticks([])
        ax_res.set_yticks([])
        ax_res.set_xlabel(f"Pred: {pred_digit}", fontsize=8, color="blue", fontweight="bold", labelpad=1)

    # 4. Side-by-Side Confusion Matrices: Clean (Before Noise) vs Noisy (After Noise)
    ax_cm_container = fig.add_subplot(grid[1, 1])
    ax_cm_container.axis("off")
    subgrid_cm = ax_cm_container.get_subplotspec().subgridspec(2, 1, hspace=0.25)

    # 4a. Clean Confusion Matrix (Before Noise)
    ax_cm1 = fig.add_subplot(subgrid_cm[0, 0])
    ax_cm1.imshow(cm_clean, cmap="Blues")
    ax_cm1.set_title(f"Before Noise (Clean Test Acc: {artifact['test_clean_accuracy']:.2f}%)", fontweight="bold", fontsize=11)
    ax_cm1.set_xlabel("Predicted")
    ax_cm1.set_ylabel("True")
    ax_cm1.set_xticks(range(10))
    ax_cm1.set_yticks(range(10))
    for r in range(10):
        for c in range(10):
            ax_cm1.text(c, r, str(cm_clean[r, c]), ha="center", va="center", fontsize=7.5,
                        color="white" if cm_clean[r, c] > cm_clean.max() / 2 else "black")

    # 4b. Noisy Confusion Matrix (After 15% Noise)
    ax_cm2 = fig.add_subplot(subgrid_cm[1, 0])
    ax_cm2.imshow(cm_noisy, cmap="Oranges")
    ax_cm2.set_title(f"After Noise (15% Noisy Test Acc: {artifact['test_accuracy']:.2f}%)", fontweight="bold", fontsize=11)
    ax_cm2.set_xlabel("Predicted")
    ax_cm2.set_ylabel("True")
    ax_cm2.set_xticks(range(10))
    ax_cm2.set_yticks(range(10))
    for r in range(10):
        for c in range(10):
            ax_cm2.text(c, r, str(cm_noisy[r, c]), ha="center", va="center", fontsize=7.5,
                        color="white" if cm_noisy[r, c] > cm_noisy.max() / 2 else "black")

    fig.suptitle(
        f"{MODEL_NAME} - {size} Images / {epochs} Epochs\n"
        f"Val Acc: {artifact['validation_accuracy']:.2f}% | Clean Test: {artifact['test_clean_accuracy']:.2f}% | 15% Noisy Test: {artifact['test_accuracy']:.2f}% (PSNR: {artifact['test_noisy_psnr']:.2f} dB)",
        fontsize=15,
        fontweight="bold"
    )

    out_path = run_folder / "all_four_charts.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"  All four charts saved to: {out_path}")
    plt.close(fig)


def plot_ten_digits(artifact, output_dir=RESULTS_DIR):
    """Plot standalone 10 digit restoration comparison for a single situation."""
    size = artifact["dataset_size"]
    epochs = artifact["epochs"]
    Xte = artifact["Xte"]
    Xnoisy = artifact["Xte_noisy"]
    recons = artifact["reconstructions"]
    yte = artifact["yte"]
    preds = artifact["predictions"]

    run_folder = output_dir / f"mnist_{size}_{epochs}"
    run_folder.mkdir(parents=True, exist_ok=True)

    sample_idx = [(yte == d).nonzero()[0].item() for d in range(10)]

    fig, axes = plt.subplots(10, 3, figsize=(6, 15))
    for ax, t in zip(axes[0], ["Original Clean", "15% Noisy Input", "Reconstructed Clean"]):
        ax.set_title(t, fontweight="bold", fontsize=10)

    for r in range(10):
        idx_i = sample_idx[r]
        pred_digit = int(preds[idx_i])
        orig_img = Xte[idx_i, 0].numpy()
        noisy_img = Xnoisy[idx_i, 0].numpy()
        recon_img = recons[idx_i, 0].numpy()

        axes[r, 0].imshow(orig_img, cmap="gray", vmin=0, vmax=1)
        axes[r, 1].imshow(noisy_img, cmap="gray", vmin=0, vmax=1)
        axes[r, 2].imshow(recon_img, cmap="gray", vmin=0, vmax=1)

        axes[r, 2].set_xlabel(f"Pred: {pred_digit}", fontsize=9, color="blue", fontweight="bold")

        for c in range(3):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])

    fig.suptitle(f"10 Digit Denoising & Prediction - {size} Images / {epochs} Epochs", fontweight="bold", fontsize=12)
    fig.tight_layout()
    out_path = run_folder / "10_digit_predictions.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_single_confusion_matrices(artifact, output_dir=RESULTS_DIR):
    """Save individual confusion matrices for Clean and Noisy test sets in the experiment folder."""
    size = artifact["dataset_size"]
    epochs = artifact["epochs"]
    cm_clean = artifact["cm_clean"]
    cm_noisy = artifact["cm_noisy"]
    run_folder = output_dir / f"mnist_{size}_{epochs}"
    run_folder.mkdir(parents=True, exist_ok=True)

    # 1. Clean confusion matrix (before noise)
    fig1, ax1 = plt.subplots(figsize=(7, 6))
    disp1 = ConfusionMatrixDisplay(cm_clean, display_labels=list(range(10)))
    disp1.plot(cmap="Blues", xticks_rotation=0, ax=ax1)
    ax1.set_title(f"Confusion Matrix Before Noise ({MODEL_NAME} - Size {size}, {epochs} Ep)\nClean Test Acc: {artifact['test_clean_accuracy']:.2f}%", fontweight="bold")
    fig1.tight_layout()
    fig1.savefig(run_folder / "confusion_matrix_clean.png", dpi=150)
    plt.close(fig1)

    # 2. Noisy confusion matrix (after noise)
    fig2, ax2 = plt.subplots(figsize=(7, 6))
    disp2 = ConfusionMatrixDisplay(cm_noisy, display_labels=list(range(10)))
    disp2.plot(cmap="Oranges", xticks_rotation=0, ax=ax2)
    ax2.set_title(f"Confusion Matrix After Noise ({MODEL_NAME} - Size {size}, {epochs} Ep)\n15% Noisy Test Acc: {artifact['test_accuracy']:.2f}%", fontweight="bold")
    fig2.tight_layout()
    fig2.savefig(run_folder / "confusion_matrix_noisy.png", dpi=150)
    plt.close(fig2)

    # 3. Combined Clean vs Noisy side-by-side
    fig3, axes3 = plt.subplots(1, 2, figsize=(14, 6))
    disp1.plot(cmap="Blues", xticks_rotation=0, ax=axes3[0], colorbar=False)
    axes3[0].set_title(f"Before Noise: Clean (Acc: {artifact['test_clean_accuracy']:.2f}%)", fontweight="bold")
    disp2.plot(cmap="Oranges", xticks_rotation=0, ax=axes3[1], colorbar=False)
    axes3[1].set_title(f"After Noise: 15% Noisy (Acc: {artifact['test_accuracy']:.2f}%)", fontweight="bold")
    fig3.suptitle(f"Confusion Matrices Before vs After Noise - Size {size}, {epochs} Epochs", fontweight="bold", fontsize=14)
    fig3.tight_layout()
    fig3.savefig(run_folder / "confusion_matrix.png", dpi=150)
    plt.close(fig3)


def plot_all_eight_confusion_matrices(artifacts_list, output_dir=RESULTS_DIR):
    """
    MASTER 8-PANEL CONFUSION MATRIX COMPARISON:
    Compares the result for every experiment before and after adding noise.
    Total: 4 experiments x 2 evaluations (Clean vs 15% Noisy) = 8 Confusion Matrices!
    """
    fig, axes = plt.subplots(4, 2, figsize=(14, 24), squeeze=False)

    for row_idx, art in enumerate(artifacts_list):
        size = art["dataset_size"]
        epochs = art["epochs"]
        cm_clean = art["cm_clean"]
        cm_noisy = art["cm_noisy"]
        clean_acc = art["test_clean_accuracy"]
        noisy_acc = art["test_accuracy"]

        # Column 0: Before Adding Noise (Clean Test Set)
        ax_clean = axes[row_idx, 0]
        disp_clean = ConfusionMatrixDisplay(cm_clean, display_labels=list(range(10)))
        disp_clean.plot(cmap="Blues", xticks_rotation=0, ax=ax_clean, colorbar=False)
        ax_clean.set_title(
            f"Exp {row_idx + 1} ({size} Images, {epochs} Ep) | BEFORE Noise (Clean Test Acc: {clean_acc:.2f}%)",
            fontweight="bold",
            fontsize=10.5
        )
        ax_clean.set_xlabel("Predicted label")
        ax_clean.set_ylabel("True label")

        # Column 1: After Adding Noise (15% Noisy Test Set)
        ax_noisy = axes[row_idx, 1]
        disp_noisy = ConfusionMatrixDisplay(cm_noisy, display_labels=list(range(10)))
        disp_noisy.plot(cmap="Oranges", xticks_rotation=0, ax=ax_noisy, colorbar=False)
        ax_noisy.set_title(
            f"Exp {row_idx + 1} ({size} Images, {epochs} Ep) | AFTER Noise (15% Noisy Test Acc: {noisy_acc:.2f}%)",
            fontweight="bold",
            fontsize=10.5
        )
        ax_noisy.set_xlabel("Predicted label")
        ax_noisy.set_ylabel("True label")

    fig.suptitle(
        f"{MODEL_NAME}: Confusion Matrices Across All 4 Situations\n(Left: Clean Test Set | Right: 15% Noisy Test Set)",
        fontsize=16,
        fontweight="bold",
        y=0.995
    )
    fig.tight_layout()
    out_path = output_dir / "confusion_matrices_all_situations.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Master 8-confusion-matrix comparison saved to: {out_path}")
    plt.close(fig)


def plot_all_ten_digits(artifacts_list, output_dir=RESULTS_DIR):
    """Plot 10 digits across all 4 experimental situations."""
    fig, axes = plt.subplots(4, 10, figsize=(20, 9))
    for row_idx, art in enumerate(artifacts_list):
        size = art["dataset_size"]
        epochs = art["epochs"]
        recons = art["reconstructions"]
        yte = art["yte"]
        preds = art["predictions"]

        sample_idx = [(yte == d).nonzero()[0].item() for d in range(10)]
        for col_idx in range(10):
            idx_i = sample_idx[col_idx]
            recon_img = recons[idx_i, 0].numpy()
            pred_digit = int(preds[idx_i])
            ax = axes[row_idx, col_idx]
            ax.imshow(recon_img, cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(f"Digit {col_idx}", fontweight="bold", fontsize=11)
            ax.set_xlabel(f"Pred: {pred_digit}", fontsize=9, color="blue", fontweight="bold")
        axes[row_idx, 0].set_ylabel(f"Size {size}\n{epochs} Ep", fontsize=11, fontweight="bold")

    fig.suptitle(f"{MODEL_NAME}: Reconstructed Digits Across All 4 Situations", fontsize=15, fontweight="bold")
    fig.tight_layout()
    out_path = output_dir / "result_10_digits_all_situations.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Multi-situation 10-digit prediction saved to: {out_path}")
    plt.close(fig)


def plot_learning_curves(artifacts_list, output_dir=RESULTS_DIR):
    """Plot multi-situation learning curves."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    # 1. Total Loss
    for i, art in enumerate(artifacts_list):
        lbl = f"Size {art['dataset_size']} - {art['epochs']} Ep"
        axes[0, 0].plot(art["history"]["epoch"], art["history"]["train_loss"], label=f"{lbl} (Train)", color=colors[i], lw=1.6)
        axes[0, 0].plot(art["history"]["epoch"], art["history"]["val_loss"], label=f"{lbl} (Val)", color=colors[i], ls="--", lw=1.6)
    axes[0, 0].set_title("Total Loss Curves", fontweight="bold")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Classification Accuracy
    for i, art in enumerate(artifacts_list):
        lbl = f"Size {art['dataset_size']} - {art['epochs']} Ep"
        axes[0, 1].plot(art["history"]["epoch"], art["history"]["train_accuracy"], label=f"{lbl} (Train)", color=colors[i], lw=1.6)
        axes[0, 1].plot(art["history"]["epoch"], art["history"]["val_accuracy"], label=f"{lbl} (Val)", color=colors[i], ls="--", lw=1.6)
    axes[0, 1].set_title("Classification Accuracy Curves", fontweight="bold")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy (%)")
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Reconstruction MSE
    for i, art in enumerate(artifacts_list):
        lbl = f"Size {art['dataset_size']} - {art['epochs']} Ep"
        axes[1, 0].plot(art["history"]["epoch"], art["history"]["train_recon_mse"], label=f"{lbl} (Train MSE)", color=colors[i], lw=1.6)
        axes[1, 0].plot(art["history"]["epoch"], art["history"]["val_recon_mse"], label=f"{lbl} (Val MSE)", color=colors[i], ls="--", lw=1.6)
    axes[1, 0].set_title("Reconstruction MSE Curves", fontweight="bold")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("MSE Loss")
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Validation PSNR
    for i, art in enumerate(artifacts_list):
        lbl = f"Size {art['dataset_size']} - {art['epochs']} Ep"
        axes[1, 1].plot(art["history"]["epoch"], art["history"]["val_psnr"], label=f"{lbl} (Val PSNR)", color=colors[i], lw=1.6)
    axes[1, 1].set_title("Validation PSNR (dB)", fontweight="bold")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("PSNR (dB)")
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)

    fig.suptitle(f"{MODEL_NAME}: Training & Validation Dynamics", fontsize=15, fontweight="bold")
    fig.tight_layout()
    out_path = output_dir / "curves.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Learning curves saved to: {out_path}")
    plt.close(fig)


def plot_final_evaluation(artifacts_list, output_dir=RESULTS_DIR):
    """Bar chart comparing validation, clean test, and 15% noisy test accuracy."""
    fig, ax = plt.subplots(figsize=(10, 6))
    labels = [f"Size {a['dataset_size']}\n{a['epochs']} Ep" for a in artifacts_list]
    x = np.arange(len(labels))
    width = 0.26

    val_accs = [a["validation_accuracy"] for a in artifacts_list]
    clean_accs = [a["test_clean_accuracy"] for a in artifacts_list]
    noisy_accs = [a["test_accuracy"] for a in artifacts_list]

    r1 = ax.bar(x - width, val_accs, width, label="Val Accuracy", color="#2ca02c")
    r2 = ax.bar(x, clean_accs, width, label="Clean Test (Before Noise)", color="#1f77b4")
    r3 = ax.bar(x + width, noisy_accs, width, label="15% Noisy Test (After Noise)", color="#ff7f0e")

    ax.set_ylabel("Accuracy (%)", fontweight="bold")
    ax.set_title(f"{MODEL_NAME}: Final Evaluation Accuracy Comparison", fontweight="bold", fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontweight="bold")
    ax.set_ylim(85, 101)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    for bar_group in [r1, r2, r3]:
        for bar in bar_group:
            h = bar.get_height()
            ax.annotate(
                f"{h:.1f}%",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8.5,
                fontweight="bold"
            )

    fig.tight_layout()
    out_path = output_dir / "final_evaluation_comparison.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Final evaluation comparison saved to: {out_path}")
    plt.close(fig)


## 8. Run All 4 Experiments, Save Visualizations & Log Epoch CSVs

Trains and evaluates the 4 situations, saving:
1. Master `results/epoch_logs.csv` (contains all epoch rows for 1–50 and 1–100)
2. Per-experiment `results/mnist_{size}_{epochs}/epoch_log.csv`
3. Summary `results/results.csv`
4. Per-experiment charts: `all_four_charts.png`, `10_digit_predictions.png`, and confusion matrices
5. Master comparisons: `confusion_matrices_all_situations.png`, `result_10_digits_all_situations.png`, `curves.png`, `final_evaluation_comparison.png`
6. Model checkpoints: `model.pth` in each experiment folder

In [29]:
def run_experiment(dataset_size, epochs):
    print("\n" + "=" * 70)
    print(f"EXPERIMENT: Size = {dataset_size} | Epochs = {epochs} | LR = {LEARNING_RATE} | Seed = {SEED}")
    print("=" * 70)
    set_seed(SEED)
    (Xtr, ytr), (Xva, yva), (Xte, yte), Xte_noisy = get_data(dataset_size, seed=SEED)
    model = get_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)
    history, best_val_acc = train_autoencoder(model, (Xtr, ytr), (Xva, yva), optimizer, epochs)
    noise_va = torch.randn(Xva.shape) * NOISE_LEVEL
    Xva_noisy = (Xva + noise_va).clamp(0.0, 1.0)
    val_res = evaluate_model(model, Xva_noisy, Xva, yva)
    clean_res = evaluate_model(model, Xte, Xte, yte)
    cm_clean = confusion_matrix(yte.numpy(), clean_res["predictions"], labels=list(range(10)))
    noisy_res = evaluate_model(model, Xte_noisy, Xte, yte)
    cm_noisy = confusion_matrix(yte.numpy(), noisy_res["predictions"], labels=list(range(10)))
    print("\nResults Summary:")
    print(f"  Validation Accuracy: {val_res['accuracy']:.2f}% | Val MSE: {val_res['mse']:.5f} | Val PSNR: {val_res['psnr']:.2f} dB")
    print(f"  Clean Test (Before Noise) -> Accuracy: {clean_res['accuracy']:.2f}% | MSE: {clean_res['mse']:.5f} | PSNR: {clean_res['psnr']:.2f} dB")
    print(f"  15% Noisy Test (After Noise) -> Accuracy: {noisy_res['accuracy']:.2f}% | MSE: {noisy_res['mse']:.5f} | PSNR: {noisy_res['psnr']:.2f} dB | SSIM: {noisy_res['ssim']:.4f}")
    
    run_folder = RESULTS_DIR / f"mnist_{dataset_size}_{epochs}"
    run_folder.mkdir(parents=True, exist_ok=True)
    safe_to_csv(pd.DataFrame(history), run_folder / "epoch_log.csv")
    
    artifact = {
        "dataset_size": dataset_size, "epochs": epochs,
        "validation_loss": 10.0 * val_res["mse"] + val_res["cls_loss"],
        "validation_accuracy": val_res["accuracy"], "validation_mse": val_res["mse"], "validation_psnr": val_res["psnr"],
        "test_clean_accuracy": clean_res["accuracy"], "test_clean_mse": clean_res["mse"], "test_clean_psnr": clean_res["psnr"],
        "test_accuracy": noisy_res["accuracy"], "test_noisy_mse": noisy_res["mse"], "test_noisy_psnr": noisy_res["psnr"], "test_noisy_ssim": noisy_res["ssim"],
        "history": history, "Xte": Xte, "yte": yte, "Xte_noisy": Xte_noisy,
        "clean_predictions": clean_res["predictions"], "predictions": noisy_res["predictions"], "reconstructions": noisy_res["reconstructions"],
        "cm_clean": cm_clean, "cm_noisy": cm_noisy, "model": model
    }
    
    # Save per-experiment plots and model checkpoint
    plot_all_four_charts(artifact, RESULTS_DIR)
    plot_ten_digits(artifact, RESULTS_DIR)
    plot_single_confusion_matrices(artifact, RESULTS_DIR)
    torch.save(model.state_dict(), run_folder / "model.pth")
    return artifact

artifacts = []
all_epoch_logs = []
for size in DATASET_SIZES:
    for epochs in EPOCHS_LIST:
        art = run_experiment(size, epochs)
        artifacts.append(art)
        for ep_i in range(epochs):
            all_epoch_logs.append({
                "size": size, "total_epochs": epochs, "epoch": art["history"]["epoch"][ep_i],
                "train_loss": art["history"]["train_loss"][ep_i], "train_recon_mse": art["history"]["train_recon_mse"][ep_i],
                "train_acc": art["history"]["train_accuracy"][ep_i], "val_loss": art["history"]["val_loss"][ep_i],
                "val_recon_mse": art["history"]["val_recon_mse"][ep_i], "val_acc": art["history"]["val_accuracy"][ep_i], "val_psnr": art["history"]["val_psnr"][ep_i]
            })

summary_data = [{
    "size": a["dataset_size"], "epochs": a["epochs"],
    "train_loss": a["history"]["train_loss"][-1], "train_recon_mse": a["history"]["train_recon_mse"][-1],
    "val_loss": a["validation_loss"], "val_acc": a["validation_accuracy"], "val_mse": a["validation_mse"], "val_psnr": a["validation_psnr"],
    "test_clean_acc": a["test_clean_accuracy"], "test_clean_mse": a["test_clean_mse"], "test_clean_psnr": a["test_clean_psnr"],
    "test_noisy_acc": a["test_accuracy"], "test_noisy_mse": a["test_noisy_mse"], "test_noisy_psnr": a["test_noisy_psnr"], "test_noisy_ssim": a["test_noisy_ssim"]
} for a in artifacts]
df_res = pd.DataFrame(summary_data)
safe_to_csv(df_res, RESULTS_DIR / "results.csv")
epoch_logs_df = pd.DataFrame(all_epoch_logs)
safe_to_csv(epoch_logs_df, RESULTS_DIR / "epoch_logs.csv")

# Save multi-situation comparison figures
plot_all_eight_confusion_matrices(artifacts, RESULTS_DIR)
plot_all_ten_digits(artifacts, RESULTS_DIR)
plot_learning_curves(artifacts, RESULTS_DIR)
plot_final_evaluation(artifacts, RESULTS_DIR)



EXPERIMENT: Size = 2500 | Epochs = 50 | LR = 0.01 | Seed = 42
  Epoch [  1/ 50] | Train Loss: 3.4185 (MSE: 0.1092, Acc: 56.23%) | Val Acc: 69.20% | Val PSNR: 8.32 dB (Best Val Acc: 69.20%)
  Epoch [ 10/ 50] | Train Loss: 0.2977 (MSE: 0.0133, Acc: 95.09%) | Val Acc: 97.20% | Val PSNR: 19.27 dB (Best Val Acc: 97.20%)
  Epoch [ 20/ 50] | Train Loss: 0.1908 (MSE: 0.0100, Acc: 97.20%) | Val Acc: 98.00% | Val PSNR: 20.56 dB (Best Val Acc: 98.80%)
  Epoch [ 30/ 50] | Train Loss: 0.1503 (MSE: 0.0091, Acc: 97.66%) | Val Acc: 98.40% | Val PSNR: 21.28 dB (Best Val Acc: 99.20%)
  Epoch [ 40/ 50] | Train Loss: 0.1262 (MSE: 0.0080, Acc: 98.91%) | Val Acc: 98.40% | Val PSNR: 21.77 dB (Best Val Acc: 99.60%)
  Epoch [ 50/ 50] | Train Loss: 0.0978 (MSE: 0.0073, Acc: 99.20%) | Val Acc: 99.20% | Val PSNR: 22.20 dB (Best Val Acc: 99.60%)

Results Summary:
  Validation Accuracy: 99.60% | Val MSE: 0.00713 | Val PSNR: 21.47 dB
  Clean Test (Before Noise) -> Accuracy: 97.80% | MSE: 0.00683 | PSNR: 21.66 dB
  

## 9. 8-Panel Master Confusion Matrix Comparison (Before vs After Noise)

In [30]:
fig_cm = plot_all_eight_confusion_matrices(artifacts, RESULTS_DIR)
plt.show()


Master 8-confusion-matrix comparison saved to: d:\rotha\AI-course\Train-Model\results\confusion_matrices_all_situations.png


## 10. Multi-Situation Learning Curves (Training & Validation Comparison)

Visualizes Total Loss, Classification Accuracy, Reconstruction MSE, and Validation PSNR
across all 4 experimental conditions (Train vs Validation).

In [31]:
plot_learning_curves(artifacts, RESULTS_DIR)
plt.show()


Learning curves saved to: d:\rotha\AI-course\Train-Model\results\curves.png


## 11. 10 Digit Denoising & Prediction Across All 4 Situations

Displays reconstructed digits (0–9) and corresponding predicted class labels for all 4 experimental conditions.

In [32]:
plot_all_ten_digits(artifacts, RESULTS_DIR)
plt.show()


Multi-situation 10-digit prediction saved to: d:\rotha\AI-course\Train-Model\results\result_10_digits_all_situations.png


## 12. Final Evaluation Accuracy Comparison

Bar chart comparing Validation Accuracy, Clean Test Accuracy (Before Noise), and 15% Noisy Test Accuracy (After Noise).

In [33]:
plot_final_evaluation(artifacts, RESULTS_DIR)
plt.show()


Final evaluation comparison saved to: d:\rotha\AI-course\Train-Model\results\final_evaluation_comparison.png


## 13. Summary Results Table & Epoch Logs Inspection

Inspects `results.csv` and `epoch_logs.csv`.

In [34]:
print("=== RESULTS SUMMARY (results.csv) ===")
display(df_res)

print("\n=== MASTER EPOCH LOGS PREVIEW (epoch_logs.csv) ===")
display(epoch_logs_df.head(10))
print(f"Total logged epochs: {len(epoch_logs_df)} (covers 50, 100, 50, 100 epochs)")


=== RESULTS SUMMARY (results.csv) ===


,size,epochs,train_loss,train_recon_mse,val_loss,val_acc,val_mse,val_psnr,test_clean_acc,test_clean_mse,test_clean_psnr,test_noisy_acc,test_noisy_mse,test_noisy_psnr,test_noisy_ssim
0,2500,50,0.097790,0.007262,0.089950,99.6,0.007130,21.469247,97.8,0.006831,21.655261,97.4,0.007165,21.447748,0.912522
1,2500,100,0.073264,0.006160,0.096153,98.8,0.007098,21.488725,97.8,0.006831,21.655261,97.4,0.007165,21.447748,0.912522
2,5000,50,0.098780,0.006351,0.083677,99.2,0.005417,22.662322,98.9,0.005044,22.972298,99.1,0.005535,22.568885,0.925022
3,5000,100,0.065798,0.005246,0.088967,98.6,0.004925,23.076369,98.9,0.004471,23.496228,99.2,0.005004,23.006506,0.929858



=== MASTER EPOCH LOGS PREVIEW (epoch_logs.csv) ===


,size,total_epochs,epoch,train_loss,train_recon_mse,train_acc,val_loss,val_recon_mse,val_acc,val_psnr
0,2500,50,1,3.418483,0.109215,56.228571,3.044425,0.147197,69.2,8.321020
1,2500,50,2,1.307643,0.048334,78.742857,0.948029,0.045960,86.8,13.376157
2,2500,50,3,0.865126,0.033136,85.771429,0.442881,0.029886,93.2,15.245328
3,2500,50,4,0.629964,0.025681,88.971429,0.335414,0.022437,94.8,16.490388
4,2500,50,5,0.518031,0.020056,90.914286,0.290114,0.018115,94.4,17.419638
5,2500,50,6,0.435826,0.017494,92.457143,0.290892,0.015766,95.6,18.022693
6,2500,50,7,0.394720,0.016092,93.542857,0.290798,0.015460,96.0,18.107989
7,2500,50,8,0.357470,0.015384,93.371429,0.209623,0.013382,97.2,18.734910
8,2500,50,9,0.328514,0.014367,94.285714,0.221328,0.012594,95.6,18.998227
9,2500,50,10,0.297741,0.013264,95.085714,0.205572,0.011841,97.2,19.265936


Total logged epochs: 300 (covers 50, 100, 50, 100 epochs)
